# Uniform state preparation

In [1]:
import numpy as np
from guppylang import guppy
from guppylang.std.builtins import array, comptime
from guppylang.std.quantum import qubit, discard_array
from guppylang.std.debug import state_output
from guppyalgos.primitives.state_preparation import uniform_state
from guppyalgos.primitives.arithmetic.comparator import comparator_ripple_cuccaro
from guppyalgos.primitives.gate_decompositions.cnx.cnx import cnx
from selene_sim import Quest
from guppyalgos.tests.helpers import switch_endianness
from guppyalgos.utils import qarray

uniform_state module is useful to preparing the uniform state of arbitrary data length L.
i.e. if log(L) is integer, we directly apply the Hadamard transformation, such as 

In [2]:
L = 4
n = int(np.ceil(np.log2(L)))
uniform = uniform_state(L)
@guppy
def main() -> None:
    """Apply main."""
    qreg = qarray(n)
    uniform(qreg)
    state_output("result_state", qreg)
    discard_array(qreg)

statevector = main.emulator(n_qubits=n).with_seed(42).run()
states = Quest.extract_states_dict(statevector.results[0].entries)
print(f"Simulated state vector: {states['result_state'].get_single_state()}")


Simulated state vector: [0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]


In [3]:
@guppy
def main() -> None:
    """Apply main."""
    qreg = qarray(n)
    uniform(qreg)
    state_output("result_state", qreg)
    discard_array(qreg)

statevector = main.emulator(n_qubits=n).with_seed(42).run()
states = Quest.extract_states_dict(statevector.results[0].entries)
print(f"Simulated state vector: {states['result_state'].get_single_state()}")

Simulated state vector: [0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]


else if log(L) is integer, we use amplified amplification to prepare, and you will need to add the comparator and cnx guppy function to combine the circuit. In this example, we use the ripple carry subtractor as the comparator, such as

In [4]:
L = 6
n = int(np.ceil(np.log2(L)))
print("The length of the data L is:", L)
print("The number of qubits used:", n)

comparator = comparator_ripple_cuccaro
uniform = uniform_state(
    L,
    cnx,
    comparator,
)


The length of the data L is: 6
The number of qubits used: 3


you then just input the prepared register into the uniform circuit

In [5]:
@guppy
def main() -> None:
    """Apply main."""
    qreg = qarray(n)
    uniform(qreg)
    state_output("result_state", qreg)
    discard_array(qreg)


when you output results, please add enough qubits as required by the `uniform_state` function

In [6]:
statevector = main.emulator(n_qubits=2 * n + 5).with_seed(42).run()
states = Quest.extract_states_dict(statevector.results[0].entries)
print(f"Simulated state vector: {switch_endianness(states['result_state'].get_single_state())}")

Simulated state vector: [0.40824829+0.j 0.40824829+0.j 0.40824829+0.j 0.40824829+0.j
 0.40824829+0.j 0.40824829+0.j 0.        +0.j 0.        +0.j]
